## 2_adversarial_debiasing.ipynb ##
## Zhang's Adversarial Debiasing
### Fairness and Compliance of AI
### University of Oklahoma

__Mitigating Disparate Impact using Zhang's Adversarial Debiasing__

This notebook demonstrates how to improve fairness metric scores by adding an adversarial debiasing network to a predictor network. We use the UCI Adult Census Income dataset to predict whether an individual earns more than $50,000/year, focusing on Sex as the protected (sensitive) attribute.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import confusion_matrix, roc_curve, auc
import torch
import torch.nn as nn
import torch.optim as optim
import random

random.seed(42)

# --- 1. Load the Dataset ---
try:
    columns = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status", 
            "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss", 
            "hours-per-week", "native-country", "income"]

    df = pd.read_csv("adult.data", names=columns, sep=r',\s*', engine='python')
    print("Dataset loaded successfully.")
except Exception as e:
    print(f"Error loading dataset: {e}")
    exit(1)

# --- 2. Feature Engineering & Simplification ---

# Simplify Workclass (Target: Max 5 categories)

# Replace the '?' with NaN values
df.replace('?', np.nan, inplace=True)

work_map = {
    'Private': 'Private',
    'Self-emp-not-inc': 'Self-Employed', 'Self-emp-inc': 'Self-Employed',
    'Local-gov': 'Government', 'State-gov': 'Government', 'Federal-gov': 'Government',
    'Without-pay': 'Other', 'Never-worked': 'Other', '?': 'Other'
}
df['work_simple'] = df['workclass'].map(work_map)

# Simplify Marital Status (Target: 3 categories)
mar_map = {
    'Married-civ-spouse': 'Married', 'Married-AF-spouse': 'Married', 'Married-spouse-absent': 'Married',
    'Divorced': 'Divorced', 'Separated': 'Divorced', 'Widowed': 'Divorced',
    'Never-married': 'Never-married'
}
df['mar_simple'] = df['marital-status'].map(mar_map)

# Targets and Sensitive Attributes
df['target'] = df['income'].apply(lambda x: 1 if '>50K' in x else 0)
df['sex_binary'] = df['sex'].apply(lambda x: 1 if x == 'Male' else 0)

# Collapse Amer-Indian-Eskimo and Other into a single category "Other" for simplicity
df['race'] = df['race'].replace({'Amer-Indian-Eskimo': 'Other', 'Other': 'Other'})

# --- 3. Define Predictor Groups ---
# Continuous: We WILL scale these
continuous_features = ['age', 'education-num', 'capital-gain', 'hours-per-week']

# Categorical/Binary: We WILL NOT scale these
categorical_features = ['work_simple', 'mar_simple', 'race']
# 'sex_binary' is also categorical, but we handle it as our sensitive attribute

# Look at the data after simplification
print(df.head())

### Transform variables
One-hot encode categorical variables and z-score normalize continuous ones. Split data into training and test sets.

In [ ]:
# --- 4. Transform Variables ---
# Create One-Hot Encodings for categories
# drop_first=True prevents the 'dummy variable trap' (multi-collinearity)
X_cat = pd.get_dummies(df[categorical_features], drop_first=True)

# Extract Numeric features
X_num = df[continuous_features]

# Create the full feature set (Adding sex_binary as a predictor as requested)
X_combined = pd.concat([X_num, X_cat, df[['sex_binary']]], axis=1).astype(float)
y = df['target']
A = df['sex_binary']

# Split the data BEFORE scaling to prevent data leakage
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X_combined, y, A, test_size=0.3, random_state=42
)

# SELECTIVE SCALING
# We only want to scale the columns that were originally continuous
scaler = StandardScaler()

# Scaling Training Data
X_train_scaled = X_train.copy()
X_train_scaled[continuous_features] = scaler.fit_transform(X_train[continuous_features])

# Scaling Testing Data
X_test_scaled = X_test.copy()
X_test_scaled[continuous_features] = scaler.transform(X_test[continuous_features])

# Note: All binary columns (Work, Marital, Race, Sex) remain exactly 0 or 1.

### Convert data to tensors
Torch requires tensors as inputs instead of dataframes

In [ ]:
# --- 5. Convert to PyTorch Tensors for the Neural Network ---
X_train_t = torch.FloatTensor(X_train_scaled.values)
y_train_t = torch.FloatTensor(y_train.values).view(-1, 1)
A_train_t = torch.FloatTensor(A_train.values).view(-1, 1)

X_test_t = torch.FloatTensor(X_test_scaled.values)
y_test_t = torch.FloatTensor(y_test.values).view(-1, 1)

### Define the neural networks
Build the predictor and adversary networks.

In [ ]:
# --- 6. Define the Model Architectures ---

# PredictorNN tries to predict the target income based on all features
class PredictorNN(nn.Module):
    def __init__(self, input_dim):
        super(PredictorNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32), # Hidden Layer
            nn.ReLU(),                # Activation
            nn.Linear(32, 1),         # Output Layer
            nn.Sigmoid()              # Probability output
        )
    def forward(self, x):
        return self.net(x)

# AdversaryNN tries to guess 'Sex' from the Predictor's output.
class AdversaryNN(nn.Module):
    """The Adversary tries to guess 'Sex' from the Predictor's output."""
    def __init__(self):
        super(AdversaryNN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

### Define fairness metrics

In [ ]:
# --- 7. Define the fairness metrics ---

def check_fairness(y_true, y_pred, sensitive_attr):
    """
    Evaluates a model's fairness by comparing performance across different groups.
    
    Parameters:
    y_true : Series/Array - The actual ground truth labels (0 or 1)
    y_pred : Series/Array - The model's binary predictions (0 or 1)
    sensitive_attr : Series/Array - The binary sensitive feature (e.g., Sex: 1=Male, 0=Female)
    """
    
    # Create a temporary dataframe to align predictions with actuals and sensitive labels
    results = pd.DataFrame({'y': y_true, 'y_hat': y_pred, 'A': sensitive_attr})
    
    # Split the results into two groups based on the sensitive attribute
    # Group 0 is often the historically 'unprivileged' or 'protected' group
    g0 = results[results['A'] == 0] 
    g1 = results[results['A'] == 1] 
    
    # Dictionary to store calculated metrics for each group
    stats = {}
    
    for name, group in [('Female', g0), ('Male', g1)]:
        # Calculate the Confusion Matrix components:
        # TN: True Negatives, FP: False Positives, FN: False Negatives, TP: True Positives
        # labels=[0,1] ensures the matrix is always 2x2 even if a group is missing a class
        tn, fp, fn, tp = confusion_matrix(group['y'], group['y_hat'], labels=[0, 1]).ravel()
        
        # 1. Demographic Parity (Statistical Independence)
        # Ratio of positive predictions regardless of the actual ground truth.
        # Ideally, P(Y_hat=1 | A=0) == P(Y_hat=1 | A=1)
        dp = group['y_hat'].mean()
        
        # 2. True Positive Rate (TPR / Sensitivity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'High Income' for those who actually have it?
        # A gap here indicates the model 'misses' qualified candidates in one group more than the other.
        tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
        
        # 3. True Negative Rate (TNR / Specificity): Part of Error Rate Parity/Separation
        # How often does the model correctly predict 'Low Income' for those who actually have it?
        # A gap here indicates one group is being 'falsely accused' of being high-income more often.
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0
        
        # 4. Predictive Parity (Sufficiency)
        # If the model predicts 'High Income', how likely is it to be correct?
        # This checks if the 'meaning' of a positive prediction is the same across groups.
        pp = tp / (tp + fp) if (tp + fp) > 0 else 0
        
        # Store metrics for this group
        stats[name] = {'DP': dp, 'TPR': tpr, 'TNR': tnr, 'PP': pp}

    # --- Print Results Table ---
    print(f"{'Metric':<22} | {'Female':<8} | {'Male':<8} | {'Difference':<10}")
    print("-" * 65)
    
    # Iterate through the metrics to calculate and display the absolute difference
    for m in ['DP', 'TPR', 'TNR', 'PP']:
        # Rename keys for the final display
        display_names = {
            'DP': 'Demographic Parity',
            'TPR': 'True Positive Rate',
            'TNR': 'True Negative Rate',
            'PP': 'Predictive Parity'
        }
        
        f_val = stats['Female'][m]
        m_val = stats['Male'][m]
        diff = abs(f_val - m_val)
        
        print(f"{display_names[m]:<22} | {f_val:.3f}   | {m_val:.3f}   | {diff:.3f}")
        
    print("-" * 65)
    print("Note: 'Error Rate Parity' is satisfied if both TPR and TNR differences are near zero.")


### Run the baseline model
Run the baseline model (no adversary) and check the results with respect to fairness

In [ ]:
# --- 8. Run 1: Baseline (No Adversary) ---
baseline_model = PredictorNN(X_train.shape[1])
optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)
criterion = nn.BCELoss()

for epoch in range(500):
    optimizer.zero_grad()
    outputs = baseline_model(X_train_t)
    loss = criterion(outputs, y_train_t)
    loss.backward()
    optimizer.step()

# Baseline FairnessResults
baseline_probs = baseline_model(X_test_t).detach().numpy()
baseline_preds = (baseline_probs > 0.5).astype(int).flatten()
print("BASELINE RESULTS:")
check_fairness(y_test, baseline_preds, A_test)

### Run the adversary model
Run the combined model, which includes the predictor and the adversary. Check the fairness results.

In [ ]:
# --- 9. Run 2: Adversarial Training ---
predictor = PredictorNN(X_train.shape[1])
adversary = AdversaryNN()
p_opt = optim.Adam(predictor.parameters(), lr=0.001)
a_opt = optim.Adam(adversary.parameters(), lr=0.001)

# Training loop with Zhang's logic
alpha = 0.5 # Hyperparameter controlling the fairness/accuracy trade-off
for epoch in range(500):
    # Predictor step
    p_opt.zero_grad()
    p_out = predictor(X_train_t)
    p_loss = criterion(p_out, y_train_t)
    
    # Adversary step (part of predictor loss)
    a_out = adversary(p_out)
    adv_loss = criterion(a_out, A_train_t)
    
    # Combined Loss: Minimize Prediction Error, Maximize Adversary Error
    total_loss = p_loss - (alpha * adv_loss)
    total_loss.backward(retain_graph=True)
    p_opt.step()
    
    # Update Adversary independently
    a_opt.zero_grad()
    a_out_final = adversary(p_out.detach())
    a_loss = criterion(a_out_final, A_train_t)
    a_loss.backward()
    a_opt.step()

# Adversarial Results
adv_probs = predictor(X_test_t).detach().numpy()
adv_preds = (adv_probs > 0.5).astype(int).flatten()
print("\nADVERSARIAL RESULTS:")
check_fairness(y_test, adv_preds, A_test)

### Check model performance
Plot the ROC curve and compare the AUC for both models

In [ ]:
# --- 9. Visualizing ROC and AUC ---
def plot_roc(y_true, b_probs, a_probs):
    fpr_b, tpr_b, _ = roc_curve(y_true, b_probs)
    fpr_a, tpr_a, _ = roc_curve(y_true, a_probs)
    
    plt.plot(fpr_b, tpr_b, label=f'Baseline AUC: {auc(fpr_b, tpr_b):.3f}')
    plt.plot(fpr_a, tpr_a, label=f'Adversarial AUC: {auc(fpr_a, tpr_a):.3f}')
    plt.plot([0, 1], [0, 1], 'k--')
    plt.legend()
    plt.title("ROC Curve Comparison: Baseline vs Adversarial")
    plt.show()

plot_roc(y_test, baseline_probs, adv_probs)